### Recuit Quantique
_______________________________________________________________________________________________________________
### Auteur : Luc Carlos Asso

### Importation des Packages

In [4]:
import numpy as np 
import dimod
from neal import SimulatedAnnealingSampler
import re

### _______________________________________________________________________________________________________________

In [298]:
class recuit_quantique :
    
    @staticmethod
    def read_instance(filename):
        with open(filename, "r") as f:
            lines = [line.strip() for line in f if line.strip()]
        n = int(re.findall(r'\d+', lines[1])[0])                                      # -------- n --------------
        profits = [[0]*n for _ in range(n)]                                           # -------- profits --------
        idx = 3
        for i in range(n):
            nums = list(map(int, re.findall(r'-?\d+', lines[idx])))
            for j in range(i+1):
                profits[i][j] = nums[j]
                profits[j][i] = nums[j]
            idx += 1

        idx += 1  # saute "#Weights:"
        weights = list(map(int, re.findall(r'-?\d+', lines[idx])))                    # -------- poids -----------
        capacity = int(re.findall(r'\d+', lines[idx+1])[0])                           # -------- capacity --------
        k = int(re.findall(r'\d+', lines[idx+2])[0])                                  # -------- k ---cardinalité-
        return n, profits, weights, capacity, k

    
    n, P, W, C, k = recuit_quantique.read_instance("Instance_60_100_25_1.txt")        #  A décocher après la prémière exécution 
    P = np.array(P)
    W = np.array(W)  
    
    @staticmethod
    def QUBO ( λ : np.ndarray) :
        QUBO = {}
        for i in range(recuit_quantique.n) :
            if (i,i) in QUBO.keys(): 
                continue
            else :
                #QUBO [(i,i)] = -recuit_quantique.P[i,i] + λ[0] * recuit_quantique.W[i]  + λ[1]
                QUBO [(i,i)] = (-recuit_quantique.P[i,i] + λ[0] * (recuit_quantique.W[i]**2 - 2 * recuit_quantique.C * recuit_quantique.W[i])
                                            + λ[1] * (1 - 2*recuit_quantique.k))
            
            
            for j in range (i+1,recuit_quantique.n) :
                 QUBO[(i,j)] = -2*recuit_quantique.P[i,j]  + 2 * λ[0] * (recuit_quantique.W[i]* recuit_quantique.W[j]) + 2 * λ[1]
        return QUBO

    @staticmethod
    def Moteur_Quantique (λ : np.ndarray) :
        bqm = dimod.BinaryQuadraticModel.from_qubo(recuit_quantique.QUBO(λ))
        sampler = SimulatedAnnealingSampler() 
        sampleset = sampler.sample( bqm, num_reads=100 )
        best = sampleset.first.sample
        energy = sampleset.first.energy
        return best,energy

    @staticmethod
    def Constraints(solution):
        selected = [ i for i in range(recuit_quantique.n) if solution[i] == 1]
        weight = sum( recuit_quantique.W[i] for i in selected)
        cardinality = len(selected)
        return weight, cardinality,selected
        
    @staticmethod
    def Relaxation_Lagrangienne( max_iter=100,alpha=0.1 ):
        lam = np.array([100 ,126])
        best_solution = None
        best_energy = float("inf")
        for t in range(max_iter):
            solution, energy = (recuit_quantique.Moteur_Quantique(lam))
            weight, card,selected = (recuit_quantique.Constraints(solution))
            g1 = weight - recuit_quantique.C   # violations
            g2 = card - recuit_quantique.k 
            lam[0] = max(0.0, lam[0] + alpha * g1)      # mise à jour des multiplicateurs
            lam[1] = lam[1] + alpha  * g2
            if energy < best_energy:             # sauvegarde meilleure solution
                best_energy = energy
                best_solution = solution
            print(f"Iteration {t}")
            print(f"Lambda = {lam}")
            print(f"Weight = {weight}")
            print(f"Cardinality = {card}")
            print(f"Energy = {energy}")
            print("-"*40)
        _,card,selected = recuit_quantique.Constraints(best_solution)

        return selected,card, best_energy
  

In [299]:
recuit_quantique.Relaxation_Lagrangienne()

Iteration 0
Lambda = [100 126]
Weight = 320
Cardinality = 18
Energy = -10272988.0
----------------------------------------
Iteration 1
Lambda = [ 99 126]
Weight = 319
Cardinality = 17
Energy = -10273097.0
----------------------------------------
Iteration 2
Lambda = [ 98 126]
Weight = 319
Cardinality = 16
Energy = -10170510.0
----------------------------------------
Iteration 3
Lambda = [ 98 126]
Weight = 321
Cardinality = 17
Energy = -10068390.0
----------------------------------------
Iteration 4
Lambda = [ 98 126]
Weight = 322
Cardinality = 16
Energy = -10068261.0
----------------------------------------
Iteration 5
Lambda = [ 98 126]
Weight = 320
Cardinality = 15
Energy = -10068053.0
----------------------------------------
Iteration 6
Lambda = [ 98 126]
Weight = 320
Cardinality = 16
Energy = -10068477.0
----------------------------------------
Iteration 7
Lambda = [ 98 126]
Weight = 320
Cardinality = 16
Energy = -10068195.0
----------------------------------------
Iteration 8
Lamb

([0, 1, 7, 10, 14, 15, 20, 24, 25, 26, 28, 32, 36, 49, 50, 54, 58],
 17,
 np.float64(-10273097.0))